# Caminhos UFV — análise reproduzível de mobilidade
Este caderno executa o mesmo núcleo de análise do painel. Comece pela demonstração com **400 participantes sintéticos**. As associações foram programadas no gerador e não descrevem a comunidade da UFV.

Execute as células em ordem em **Ambiente de execução → Executar tudo**. Na primeira instalação, se solicitado, use **Reiniciar sessão** e execute tudo novamente. O Colab usa versões fixadas compatíveis com seus pacotes pré-instalados; elas podem diferir do servidor, embora os algoritmos sejam os mesmos. A trilha de auditoria registra o ambiente efetivamente utilizado. A fonte padrão dispensa login. Para a planilha privada, altere FONTE e autorize sua própria conta; o caderno não recebe senhas nem chaves de serviço. O notebook público é distribuído sem saídas de execução.

## Perguntas que o código pode investigar
1. Como perfis e modos de transporte se distribuem no recorte?
2. Quais modelos classificam o uso **atual** de bicicleta fora das amostras de treinamento?
3. Há associação indireta exploratória entre infraestrutura, segurança e intenção?
4. A associação entre tipo de via e escolha hipotética varia com a confiança?

Não são estimados efeitos causais, disposição a pagar isolada ou impacto de uma intervenção. CFA, mixed logit e acompanhamento longitudinal exigem desenvolvimentos adicionais.

In [ ]:
# Ambiente fixado para permitir comparação. Pode levar alguns minutos na primeira execução.
%pip -q install numpy==2.2.6 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.7.2 statsmodels==0.14.5 matplotlib==3.10.3 networkx==3.5 gspread==6.2.1 folium==0.20.0 ipywidgets==8.1.7
import sys, importlib.metadata as _metadata
_loaded = [(name, getattr(sys.modules[name], "__version__", ""), _metadata.version(name)) for name in ["numpy", "pandas", "scipy"] if name in sys.modules]
if any(loaded != installed for name, loaded, installed in _loaded):
    raise RuntimeError("Instalação concluída. No menu Ambiente de execução, escolha Reiniciar sessão; depois Executar tudo novamente. Isso recarrega as bibliotecas já abertas pelo Colab.")

## 1 Escolha a origem e configure o recorte
**sintetica** baixa somente a demonstração pública e verifica SHA256. **google_sheets** faz leitura autenticada da aba indicada, com coluna `response_json` contendo cada resposta no formato versão 3. **arquivo** aceita JSON exportado pelo painel. A planilha legada versão 2 não é convertida silenciosamente; é preciso usar uma aba versão 3.

Os filtros são aplicados antes de todas as análises. Datas são inclusivas no horário de São Paulo. Nenhuma rotina grava ou exclui linhas na planilha. A leitura no Colab processa dados na infraestrutura Google; use dados reais apenas no ambiente autorizado pelo projeto e não compartilhe o notebook com saídas individuais.

In [ ]:
FONTE = "sintetica" # "sintetica", "google_sheets" ou "arquivo"
PLANILHA_ID = "" # preencha somente em sua cópia privada
ABA = "respostas" # coluna response_json; uma resposta versão 3 por linha
ARQUIVO_JSON = "" # caminho local; vazio abre seletor no Colab
FILTRO_VINCULO = "" # vazio = todos; consulte categorias na célula de qualidade
FILTRO_MODO = ""
DATA_INICIO = "" # AAAA-MM-DD, inclusive
DATA_FIM = "" # AAAA-MM-DD, inclusive

In [ ]:
import json, hashlib, urllib.request, sys, platform
from pathlib import Path
BASE_PUBLICA = "https://raw.githubusercontent.com/alexandrerodrigues2311/caminhos-ufv-ciencia-aberta/main/"
SHA256_SINTETICA = "75a6f7a41dd347451acbcf73cc6d0ac4077e0b3006172d3b921cc47af7a3a4d2"
if FONTE == "sintetica":
    raw = urllib.request.urlopen(BASE_PUBLICA + "simulacao-400.json", timeout=30).read()
    if hashlib.sha256(raw).hexdigest() != SHA256_SINTETICA:
        raise ValueError("A versão da base mudou. Obtenha o notebook da mesma versão; não ignore esta verificação.")
    payload = json.loads(raw)
elif FONTE == "google_sheets":
    if not PLANILHA_ID.strip(): raise ValueError("Informe o ID da planilha na sua cópia privada.")
    from google.colab import auth
    import google.auth, gspread
    auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
    worksheet = gspread.authorize(credentials).open_by_key(PLANILHA_ID).worksheet(ABA)
    values = worksheet.get_all_values()
    if not values or "response_json" not in values[0]:
        raise ValueError("A aba deve conter o cabeçalho response_json.")
    col = values[0].index("response_json")
    payload = [json.loads(row[col]) for row in values[1:] if len(row)>col and row[col].strip()]
elif FONTE == "arquivo":
    if not ARQUIVO_JSON:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1: raise ValueError("Selecione um único JSON exportado pelo painel.")
        payload = json.loads(next(iter(uploaded.values())))
    else:
        payload = json.loads(Path(ARQUIVO_JSON).read_text(encoding="utf-8-sig"))
else:
    raise ValueError("Fonte desconhecida.")
records = payload.get("records") if isinstance(payload,dict) else payload

## 2 Leia as funções de preparação e análise
O código abaixo está incorporado ao notebook e também disponível em `analises.py`. Nenhum código remoto é executado. `frames` cria as variáveis; `ml` mantém a imputação dentro das dobras; `mediation` reamostra pessoas; `panel` trata as nove decisões como dependentes dentro de cada participante.

Idade, confiança, infraestrutura, segurança, vínculo, renda e habilidade são preditores de ML. Dias de uso e intenção não entram como preditores. q43 é índice operacional de componentes, sem alegação de escala latente validada. q44 e intenção também exigem validação e pré-teste antes da interpretação empírica.

In [ ]:
"""Análises reproduzíveis. Sem gravação de respostas ou coordenadas em logs."""
import json,math,warnings,hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import balanced_accuracy_score,f1_score,log_loss,brier_score_loss,confusion_matrix
import statsmodels.api as sm

SEED=8421264
def clean(v):
 if isinstance(v,dict):return {str(k):clean(x) for k,x in v.items()}
 if isinstance(v,(list,tuple,np.ndarray)):return [clean(x) for x in v]
 if isinstance(v,(np.integer,)):return int(v)
 if isinstance(v,(float,np.floating)):return float(v) if np.isfinite(v) else None
 return v
def score(a):
 return float(a) if isinstance(a,(int,float)) and not isinstance(a,bool) and np.isfinite(a) else np.nan
def meanitems(a,q):
 vals=[score(a.get(f'{q}_i{i}')) for i in [1,2,3]]
 return float(np.mean(vals)) if all(np.isfinite(vals)) else np.nan

In [ ]:
def frames(records):
 conf=['Nada confiante','Pouco confiante','Moderadamente confiante','Muito confiante','Totalmente confiante']
 intent=['Muito baixa','Baixa','Moderada','Alta','Muito alta']
 rows=[]
 for r in records:
  a=r['answers'];rows.append(dict(id=r['response_id'],age=score(a.get('q2')),role=a.get('q3'),income=a.get('q5'),skill=a.get('q10'),confidence=conf.index(a['q10a'])+1 if a.get('q10a') in conf else np.nan,infra=meanitems(a,'q43'),safety=meanitems(a,'q44'),intent=intent.index(a['q42'])+1 if a.get('q42') in intent else np.nan,days=score(a.get('q41')),mode=a.get('q16')))
 return pd.DataFrame(rows)

In [ ]:
def ml(records):
 df=frames(records).dropna(subset=['mode']);df=df[df['mode']!=''];df['target']=df['mode'].isin(['Bicicleta comum','Bicicleta elétrica']).astype(int)
 if len(df)<80 or df.target.value_counts().min()<20 or df.target.nunique()<2:raise ValueError('ML: são necessários 80 casos e pelo menos 20 pessoas por classe.')
 numeric=['age','confidence','infra','safety'];categorical=['role','income','skill'];cols=numeric+categorical
 X=df[cols].copy().replace({None:np.nan});y=df.target.to_numpy();groups=df.id.to_numpy()
 pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),numeric),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),categorical)])
 models={'Referência de prevalência':DummyClassifier(strategy='prior'),'Logística regularizada':LogisticRegression(C=1,max_iter=1000,random_state=SEED),'Random forest':RandomForestClassifier(n_estimators=100,max_depth=4,min_samples_leaf=8,random_state=SEED,n_jobs=1),'Gradient boosting':GradientBoostingClassifier(n_estimators=60,max_depth=2,min_samples_leaf=10,learning_rate=.05,random_state=SEED)}
 folds=list(StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED).split(X,y,groups));out=[]
 for name,model in models.items():
  prob=np.zeros(len(df));fold_metrics=[]
  for train,test in folds:
   if set(groups[train])&set(groups[test]):raise ValueError('Vazamento entre participantes detectado.')
   pipe=Pipeline([('prepare',pre),('model',model)]);pipe.fit(X.iloc[train],y[train]);prob[test]=pipe.predict_proba(X.iloc[test])[:,1]
   fold_metrics.append(float(balanced_accuracy_score(y[test],prob[test]>=.5)))
  pred=prob>=.5;cal=[]
  for low in np.arange(0,1,.2):
   ix=(prob>=low)&(prob<low+.2 if low<.8 else prob<=1)
   if ix.sum():cal.append(dict(predicted=float(prob[ix].mean()),observed=float(y[ix].mean()),n=int(ix.sum())))
  out.append(dict(model=name,balanced_accuracy=balanced_accuracy_score(y,pred),macro_f1=f1_score(y,pred,average='macro'),log_loss=log_loss(y,prob,labels=[0,1]),brier=brier_score_loss(y,prob),confusion=confusion_matrix(y,pred,labels=[0,1]).tolist(),fold_balanced=fold_metrics,calibration=cal))
 best=min(out[1:],key=lambda v:v['log_loss']);base=out[0]
 return dict(n=len(df),excluded=len(records)-len(df),target='Bicicleta comum ou elétrica no modo atual versus outros meios',features=cols,folds=5,models=out,interpretation=f"O menor log loss entre os modelos candidatos foi obtido por {best['model']}. "+('Superou' if best['log_loss']<base['log_loss'] else 'Não superou')+' a referência de prevalência neste conjunto. A seleção pelo mesmo desempenho de validação é exploratória; requer confirmação em amostra independente.',note='Hiperparâmetros fixados antes do ajuste; imputação e codificação dentro de cada dobra. Dias de uso e intenção foram excluídos dos preditores para evitar proximidade excessiva com o alvo. Não prevê adoção futura. Dispersão entre dobras não é intervalo de confiança.')

In [ ]:
def mediation(records,boot=500):
 d=frames(records).dropna(subset=['age','infra','safety','intent']);n=len(d)
 if n<80:raise ValueError('Mediação: são necessários 80 casos completos nos blocos de infraestrutura, segurança, intenção e idade.')
 x=d.infra.to_numpy();m=d.safety.to_numpy();y=d.intent.to_numpy();age=(d.age.to_numpy()-30)/10
 def fit(ix,rank=False):
  xx=x[ix];mm=m[ix];yy=rankdata(y[ix]) if rank else y[ix];aa=age[ix]
  A=np.column_stack([np.ones(len(ix)),xx,aa]);B=np.column_stack([np.ones(len(ix)),xx,mm,aa]);C=np.column_stack([np.ones(len(ix)),xx,aa])
  if np.linalg.matrix_rank(A)<3 or np.linalg.matrix_rank(B)<4:raise ValueError('Sem variação suficiente para mediação.')
  a=np.linalg.lstsq(A,mm,rcond=None)[0][1];b=np.linalg.lstsq(B,yy,rcond=None)[0];total=np.linalg.lstsq(C,yy,rcond=None)[0][1]
  return np.array([a,b[2],a*b[2],b[1],total])
 values=fit(np.arange(n));rng=np.random.default_rng(SEED);draw=[]
 for _ in range(boot):
  try:draw.append(fit(rng.integers(0,n,n)))
  except ValueError:pass
 if len(draw)<boot*.9:raise ValueError('Mediação instável nas reamostragens. Amplie ou revise o recorte.')
 ci=np.percentile(draw,[2.5,97.5],axis=0)
 estimates=[dict(term=term,estimate=values[j],low=ci[0,j],high=ci[1,j]) for j,term in enumerate(['a: infraestrutura → segurança','b: segurança → intenção','Associação indireta a×b','Associação direta','Associação total'])]
 return dict(n=n,excluded=len(records)-n,bootstrap=len(draw),estimates=estimates,rank_sensitivity_indirect=fit(np.arange(n),True)[2],note='Regressões lineares exploratórias dos escores médios completos q43 e q44 e intenção ordinal q42, ajustadas por idade. Bootstrap por pessoa. q43 é um resumo de componentes, não uma escala latente validada. A sensibilidade por postos usa outra unidade e serve para conferir o sinal. Dados transversais não identificam mediação causal; confundimento, seleção e temporalidade permanecem.',interpretation='O intervalo da associação indireta '+('inclui zero; não há direção estatisticamente definida neste ajuste.' if ci[0,2]<=0<=ci[1,2] else 'não inclui zero neste ajuste exploratório. Isso não comprova mecanismo causal.'))

In [ ]:
def panel(records,schema):
 df=frames(records).set_index('id');qs=[q for q in schema['questions'] if q['type']=='scenario'];roads=list(dict.fromkeys(q['road'] for q in qs));slopes=list(dict.fromkeys(q['slope'] for q in qs));bikes=list(dict.fromkeys(q['bike'] for q in qs));rows=[]
 for r in records:
  c=df.loc[r['response_id'],'confidence']
  if not np.isfinite(c):continue
  for q in qs:
   v=r['answers'].get(q['id'])
   if v not in q['options']:continue
   row=dict(id=r['response_id'],choice=int(v==q['options'][0]),time=(float(q['time'].split()[0])-15)/5,confidence=c-3)
   for i,label in enumerate(roads[1:]):row[f'via_{i+1}']=int(q['road']==label)
   for i,label in enumerate(slopes[1:]):row[f'relevo_{i+1}']=int(q['slope']==label)
   for i,label in enumerate(bikes[1:]):row[f'pacote_{i+1}']=int(q['bike']==label)
   row['via_x_confianca']=row.get('via_2',row.get('via_1',0))*(c-3);rows.append(row)
 d=pd.DataFrame(rows)
 if d.empty or d.id.nunique()<60 or d.choice.nunique()<2 or d.choice.value_counts().min()<30:raise ValueError('Painel: são necessários 60 participantes e 30 escolhas em cada alternativa.')
 x=sm.add_constant(d.drop(columns=['id','choice']).astype(float),has_constant='add')
 if np.linalg.matrix_rank(x)<x.shape[1]:raise ValueError('O desenho não identifica todos os termos neste recorte.')
 with warnings.catch_warnings():
  warnings.simplefilter('ignore');fit=sm.GEE(d.choice,x,groups=d.id,family=sm.families.Binomial(),cov_struct=sm.cov_struct.Exchangeable()).fit(maxiter=100)
 if not fit.converged or not np.isfinite(fit.params).all() or not np.isfinite(fit.bse).all():raise ValueError('GEE não convergiu de forma estável. Revise o recorte.')
 ci=fit.conf_int();terms=[dict(term=k,beta=fit.params[k],low=ci.loc[k,0],high=ci.loc[k,1],odds=np.exp(fit.params[k])) for k in x.columns]
 # Probabilidades padronizadas com demais atributos fixos nas linhas observadas.
 preds=[]
 for c in [-2,0,2]:
  for road in [0,1]:
   xx=x.copy();xx['confidence']=c
   for col in [v for v in xx.columns if v.startswith('via_') and v!='via_x_confianca']:xx[col]=0
   xx['via_2' if 'via_2' in xx else 'via_1']=road;xx['via_x_confianca']=road*c
   preds.append(dict(confidence=c+3,protected=road,probability=float(fit.predict(xx).mean())))
 return dict(people=d.id.nunique(),tasks=len(d),coefficients=terms,predictions=preds,working_correlation=float(fit.cov_struct.dep_params),references=dict(road=roads[0],slopes=slopes,bikes=bikes,roads=roads),note='GEE logístico marginal com correlação de trabalho permutável e covariância robusta por participante. Não é mixed logit e não estima distribuição de preferências. Interação pré-especificada entre a terceira categoria de via e confiança (q10a). Preço e bicicleta seguem pacotes do desenho original; não há disposição a pagar isolada.',interpretation='A interação descreve diferenças na associação entre via e escolha conforme a confiança. Compare probabilidades padronizadas; não interprete odds ratio como variação percentual direta da probabilidade.')

In [ ]:
def run(payload):
 records=payload.get('records',[])
 if not isinstance(records,list) or not 1<=len(records)<=2000:raise ValueError('Selecione entre 1 e 2.000 participantes.')
 ids=[r.get('response_id') for r in records]
 if len(set(ids))!=len(ids):raise ValueError('Códigos duplicados: análise interrompida.')
 if any(r.get('schema_version')!='ufv-mobilidade-3.0.0' or not isinstance(r.get('answers'),dict) for r in records):raise ValueError('Use exclusivamente a versão 3 do instrumento.')
 kinds={('synthetic' if r.get('is_synthetic') else 'demo' if r.get('is_demo') else 'real') for r in records}
 if len(kinds)!=1:raise ValueError('Separe dados reais e testes.')
 method=payload.get('method');schema={'questions': [{'id': 'q28', 'number': 28, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Na rua, junto com os carros', 'time': '10 min', 'slope': 'Plano', 'bike': 'Própria (ou emprestada) – grátis', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q29', 'number': 29, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Na rua, junto com os carros', 'time': '15 min', 'slope': 'Subidas moderadas', 'bike': 'Compartilhada comum – R$ 1,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q30', 'number': 30, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Na rua, junto com os carros', 'time': '20 min', 'slope': 'Subidas fortes', 'bike': 'Compartilhada elétrica – R$ 2,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q31', 'number': 31, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclofaixa pintada', 'time': '10 min', 'slope': 'Subidas moderadas', 'bike': 'Compartilhada elétrica – R$ 2,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q32', 'number': 32, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclofaixa pintada', 'time': '15 min', 'slope': 'Subidas fortes', 'bike': 'Própria (ou emprestada) – grátis', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q33', 'number': 33, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclofaixa pintada', 'time': '20 min', 'slope': 'Plano', 'bike': 'Compartilhada comum – R$ 1,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q34', 'number': 34, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclovia separada por meio-fio', 'time': '10 min', 'slope': 'Subidas fortes', 'bike': 'Compartilhada comum – R$ 1,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q35', 'number': 35, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclovia separada por meio-fio', 'time': '15 min', 'slope': 'Plano', 'bike': 'Compartilhada elétrica – R$ 2,00', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}, {'id': 'q36', 'number': 36, 'type': 'scenario', 'title': 'Como você viria ao campus nesta situação?', 'road': 'Ciclovia separada por meio-fio', 'time': '20 min', 'slope': 'Subidas moderadas', 'bike': 'Própria (ou emprestada) – grátis', 'options': ['Bicicleta nas condições apresentadas', 'Manteria meu deslocamento habitual'], 'hint': 'Os atributos descrevem um pacote. O custo se refere a uma ida. A manutenção do deslocamento habitual inclui as condições já relatadas.'}]}
 if method=='ml':result=ml(records)
 elif method=='mediation':result=mediation(records)
 elif method=='panel':result=panel(records,schema)
 else:raise ValueError('Método inválido.')
 return clean(dict(method=method,kind=next(iter(kinds)),seed=SEED,result=result,fingerprint=hashlib.sha256(json.dumps(records,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:16],warning='SIMULAÇÃO: nenhum resultado descreve a população de Viçosa.' if kinds=={'synthetic'} else 'Associações da amostra; não implicam causalidade ou representatividade.'))

## 3 Valide a base e aplique filtros
Duplicações, mistura de simulação e pesquisa, incompatibilidade de instrumento e datas inválidas interrompem a execução. Ausência não vira zero. Campos de endereço, coordenadas e texto livre não entram no núcleo dos modelos. Confira as categorias antes de filtrar; recortes pequenos podem não identificar os modelos.

In [ ]:
import pandas as pd
if not isinstance(records,list) or not records: raise ValueError("A base não contém respostas.")
if any(not isinstance(r,dict) or not isinstance(r.get('answers'),dict) for r in records): raise ValueError("Estrutura inválida.")
if any(r.get('schema_version') != 'ufv-mobilidade-3.0.0' for r in records): raise ValueError("Use a versão 3 do instrumento.")
ids=[r.get('response_id') for r in records]
if any(not isinstance(i,str) or not i.strip() for i in ids) or len(set(ids))!=len(ids): raise ValueError("IDs ausentes ou duplicados.")
kinds={('synthetic' if r.get('is_synthetic') else 'demo' if r.get('is_demo') else 'real') for r in records}
if len(kinds)!=1: raise ValueError("Não misture respostas reais e de teste.")
NATUREZA=next(iter(kinds))
if FONTE=='sintetica' and NATUREZA!='synthetic': raise ValueError("Demonstração deve conter somente dados sintéticos.")
all_frames=frames(records)
print('Natureza:',NATUREZA,'| participantes:',len(records))
print('Vínculos:',sorted(all_frames.role.dropna().unique()))
print('Modos:',sorted(all_frames['mode'].dropna().unique()))
def as_day(value):
    return pd.to_datetime(value,utc=True,errors='raise').tz_convert('America/Sao_Paulo').date()
start=pd.Timestamp(DATA_INICIO).date() if DATA_INICIO else None
end=pd.Timestamp(DATA_FIM).date() if DATA_FIM else None
if start and end and start>end: raise ValueError('Datas invertidas.')
selected=[]
for r in records:
    a=r['answers']
    if FILTRO_VINCULO and a.get('q3')!=FILTRO_VINCULO: continue
    if FILTRO_MODO and a.get('q16')!=FILTRO_MODO: continue
    if start or end:
        day=as_day(r.get('completed_at'))
        if (start and day<start) or (end and day>end): continue
    selected.append(r)
if not selected: raise ValueError('O recorte não contém participantes.')
safe_records=[{**r,'answers':{k:v for k,v in r['answers'].items() if k not in ['q11','q14','q15','q40']}} for r in selected]
df=frames(safe_records)
display(df.isna().mean().mul(100).rename('Ausência percentual').to_frame())
print('Participantes no recorte:',len(df))

## 4 Descreva a amostra antes de ajustar modelos
Compare os perfis da amostra com dados institucionais agregados, quando disponíveis. Aderência voluntária e coleta online não constituem amostragem probabilística. O gráfico não demonstra representatividade. Nas bases reais, categorias com menos de cinco pessoas são omitidas da visualização; isso reduz exposição, mas não é garantia de anonimização.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})
counts=df['mode'].value_counts().sort_values()
shown=counts if NATUREZA=='synthetic' else counts[counts>=5]
fig,ax=plt.subplots(figsize=(9,4.5))
ax.barh(shown.index,shown.values,color='#901812')
ax.set(xlabel='Participantes',title=f'Meio de transporte atual • {NATUREZA} • n={len(df)}')
plt.tight_layout();plt.show()
results={}

## 5 Compare quatro modelos de machine learning
Validação cruzada estratificada em cinco dobras com agrupamento por pessoa, parâmetros pré-fixados e semente registrada. Menor log loss e Brier são melhores; acurácia balanceada e macro F1 ajudam com classes desiguais. O melhor resultado no mesmo conjunto de validação é exploratório, sem garantia de generalização. Os limites mínimos de casos são proteções computacionais, não cálculo de poder. Se uma etapa falhar por recorte insuficiente, amplie o recorte antes de interpretar.

In [ ]:
results['ml']=run({'method':'ml','records':safe_records})
metrics=pd.DataFrame(results['ml']['result']['models'])
display(metrics[['model','balanced_accuracy','macro_f1','log_loss','brier']])
print(results['ml']['result']['interpretation'])
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].barh(metrics.model,metrics.balanced_accuracy,color='#901812')
axes[0].set(xlim=(0,1),xlabel='Acurácia balanceada')
for model in results['ml']['result']['models']:
    c=model['calibration']
    axes[1].plot([v['predicted'] for v in c],[v['observed'] for v in c],marker='o',label=model['model'])
axes[1].plot([0,1],[0,1],'k--',alpha=.4)
axes[1].set(xlabel='Probabilidade prevista',ylabel='Frequência observada',title='Calibração fora do treino')
axes[1].legend(fontsize=7);plt.tight_layout();plt.show()

## 6 Estime a associação indireta exploratória
Infraestrutura → segurança → intenção, ajustada por idade; 500 reamostragens por pessoa. Escores e intenção são aproximados por regressão linear. A análise por postos verifica o sinal, com unidade diferente. Intervalos que não incluem zero **não comprovam mediação causal**: não há temporalidade nem controle suficiente de confundimento. Casos completos podem selecionar participantes.

In [ ]:
results['mediation']=run({'method':'mediation','records':safe_records})
m=results['mediation']['result'];display(pd.DataFrame(m['estimates']))
print('Casos:',m['n'],'Excluídos:',m['excluded'],'Bootstrap:',m['bootstrap'])
print(m['interpretation']);print(m['note'])

## 7 Ajuste escolhas repetidas e moderação
GEE logístico com correlação de trabalho permutável e covariância robusta por pessoa. O termo de interação usa a terceira categoria de via e confiança. Compare probabilidades padronizadas, além dos coeficientes. Não é mixed logit; os preços permanecem combinados com tipos de bicicleta no desenho original e não permitem estimar disposição a pagar isolada.

In [ ]:
results['panel']=run({'method':'panel','records':safe_records})
p=results['panel']['result'];display(pd.DataFrame(p['coefficients']))
pred=pd.DataFrame(p['predictions'])
fig,ax=plt.subplots(figsize=(7,4))
for flag,label in [(0,'Via de referência'),(1,'Ciclovia separada por meio-fio')]:
    part=pred[pred.protected==flag]
    ax.plot(part.confidence,part.probability,marker='o',label=label)
ax.set(xlabel='Confiança (1–5)',ylabel='Probabilidade padronizada',ylim=(0,1),title=f'Escolhas hipotéticas • {NATUREZA}')
ax.legend();plt.tight_layout();plt.show()
print(p['note'])

## 8 Visualize redes agregadas de origem e transporte
As arestas são contagens de respostas, não trajetos reais ou relações causais. Não se usam ruas ou coordenadas exatas. Em dados reais, ligações com menos de cinco pessoas são omitidas. Para mapas territoriais, o painel apresenta células aproximadas e malha municipal IBGE; cobertura e viés espacial precisam ser avaliados antes de qualquer generalização.

In [ ]:
import networkx as nx
from collections import Counter
pairs=Counter((r['answers'].get('q11',{}).get('neighborhood'),r['answers'].get('q16')) for r in selected if isinstance(r['answers'].get('q11'),dict))
G=nx.Graph()
for (origin,mode),n in pairs.items():
    if not origin or not mode or (NATUREZA!='synthetic' and n<5): continue
    G.add_edge('Origem: '+origin,'Modo: '+mode,weight=n)
if len(G):
    fig,ax=plt.subplots(figsize=(12,7))
    pos=nx.spring_layout(G,seed=SEED,weight='weight')
    nx.draw_networkx(G,pos,ax=ax,font_size=7,node_size=200,node_color=['#d1a705' if x.startswith('Modo:') else '#b97976' for x in G],width=[.5+G[u][v]['weight']**.5 for u,v in G.edges],edge_color='#ccc1b4')
    ax.set_title(f'Rede agregada de origem e modo • {NATUREZA}');ax.axis('off');plt.tight_layout();plt.show()
else: print('Sem ligações suficientes para exibição agregada.')

## 8A Mapa real de Viçosa, UFV e cada deslocamento
O mapa utiliza ruas reais e referências documentadas da universidade. Pontos de resposta preservam as coordenadas disponíveis; uma linha reta **não é a rota percorrida**. Destinos fictícios A/B/C não são endereços. A opção ilustrativa, desativada por padrão, demonstra a ligação com Biblioteca Central, PVA ou Edifício Arthur Bernardes e só funciona para registros sintéticos. Desconhecidos permanecem sem ponto.

Dados reais devem permanecer em uma sessão privada autorizada. O mapa individual contém informação locacional: não publique seu HTML nem as saídas. O provedor de imagens cartográficas recebe a área visualizada, mas nenhum endereço é enviado a geocodificadores ou serviços de rotas. As ruas vetoriais permitem exibir a malha caso as imagens não carreguem. OSM: dados colaborativos, não cadastrais; atribuição e licença ODbL mantidas.

In [ ]:
# Cópias cartográficas públicas verificadas; nenhuma consulta usa endereços de participantes.
CARTOGRAPHY_HASHES={'ufv-places.json': 'be346e4811223e71962ca194d3bf271a809b3a9fd880663a727627334c4f0d37', 'ufv-campus.geojson': '07dcc31caa704aa7b6ab4444f30b12c01ccb27be67ece42f46893576d0320f87', 'vicosa-streets.geojson': '761390c9c9c6afcdccf4feb01b9c3f373076fd852076744c3282cb236d078a3f', 'questionario.schema.json': 'c879b8beb1961639f358474bdfcd4762eea4ee33a3fb8a623ef00b1d0a0b4a49'}
cartography={}
for name in ['ufv-places.json','ufv-campus.geojson','vicosa-streets.geojson']:
    content=urllib.request.urlopen(BASE_PUBLICA+name,timeout=45).read()
    if hashlib.sha256(content).hexdigest()!=CARTOGRAPHY_HASHES[name]:
        raise ValueError('Cartografia alterada: atualize a versão do notebook antes de prosseguir.')
    cartography[name]=json.loads(content)
catalog=cartography['ufv-places.json'];campus=cartography['ufv-campus.geojson'];streets=cartography['vicosa-streets.geojson']
print('Referências reais da UFV:',len(catalog['places']),'· segmentos da malha:',len(streets['features']))

In [ ]:
"""Mapa real, inspeção de respostas e proveniência explícita dos destinos."""
import math, unicodedata, html
import folium

def valid_point(p):
    return isinstance(p,dict) and all(isinstance(p.get(k),(float,int)) and not isinstance(p[k],bool) and math.isfinite(p[k]) and abs(p[k])<=bound for k,bound in [('lat',90),('lng',180)])

def destination_for(record, places, illustrative=False):
    a=record['answers']
    if valid_point(a.get('q15_point')):
        return dict(a['q15_point'],name=a.get('q15') or 'Destino informado',provenance='Coordenada informada na resposta')
    norm=lambda x: ''.join(c for c in unicodedata.normalize('NFD',str(x or '')) if unicodedata.category(c)!='Mn').strip().lower()
    aliases={'pva':'8222713258','pavilhao de aulas i':'8222713258','pvb':'21450486','bct':'292666798','biblioteca central':'292666798','bernardao':'148819551','reitoria':'148819551'}
    name=norm(a.get('q15'))
    for p in places:
        if norm(p['name'])==name or p['id']==aliases.get(name):
            return dict(p,provenance='Nome associado ao catálogo OpenStreetMap')
    if illustrative and record.get('is_synthetic') is True:
        chosen={'Destino fictício A':'292666798','Destino fictício B':'8222713258','Destino fictício C':'148819551'}.get(a.get('q15'))
        for p in places:
            if p['id']==chosen:return dict(p,illustrative=True,provenance='ATRIBUIÇÃO ILUSTRATIVA. Não é um destino real informado na resposta.')
    return None

def territorial_map(records, catalog, campus, response_id='', illustrative=False, streets=None):
    """Nunca geocodifica endereços nem envia respostas a um serviço de rotas."""
    m=folium.Map(location=[-20.758,-42.875],zoom_start=14,tiles='OpenStreetMap',control_scale=True,scrollWheelZoom=False)
    folium.GeoJson(campus,name='Campus UFV — contorno colaborativo',style_function=lambda _:dict(color='#a37900',weight=3,fillColor='#d1a705',fillOpacity=.15),tooltip='UFV · contorno OSM, não cadastral').add_to(m)
    if streets:
        folium.GeoJson(streets,name='Ruas vetoriais OSM (funcionam sem imagens do mapa)',show=False,style_function=lambda _:dict(color='#8a8a82',weight=1)).add_to(m)
    landmarks=folium.FeatureGroup(name='Referências reais da UFV').add_to(m)
    for p in catalog['places']:
        folium.CircleMarker([p['lat'],p['lng']],radius=5,color='#8e6c00',fill=True,fill_opacity=1,fill_color='#d1a705',tooltip=html.escape(p['name'])).add_to(landmarks)
    origins=folium.FeatureGroup(name='Origens no recorte').add_to(m)
    for r in records:
        a=r['answers'];o=(a.get('q11') or {}).get('point')
        if not valid_point(o):continue
        popup='<b>'+html.escape(r['response_id'])+'</b><br>'+html.escape(str(a.get('q16') or 'Meio não informado'))+'<br>Destino original: '+html.escape(str(a.get('q15') or 'Não informado'))
        folium.CircleMarker([o['lat'],o['lng']],radius=4,color='#901812',weight=1,fill=True,fill_opacity=.65,popup=folium.Popup(popup,max_width=280),tooltip=html.escape(r['response_id'])).add_to(origins)
    chosen=next((r for r in records if r['response_id']==response_id),None)
    if chosen:
        a=chosen['answers'];o=(a.get('q11') or {}).get('point');d=destination_for(chosen,catalog['places'],illustrative);points=[]
        if valid_point(o):
            points.append([o['lat'],o['lng']]);folium.CircleMarker(points[-1],radius=10,color='#901812',weight=4,tooltip='Origem selecionada').add_to(m)
        if d:
            points.append([d['lat'],d['lng']]);folium.CircleMarker(points[-1],radius=9,color='#246480',weight=4,tooltip=html.escape(d['name']),popup=html.escape(d['provenance'])).add_to(m)
        if len(points)==2:folium.PolyLine(points,color='#246480',dash_array='7 9',tooltip='Ligação origem–destino; não é rota percorrida').add_to(m)
        if points:m.fit_bounds(points,max_zoom=16,padding=(35,35))
    folium.LayerControl(collapsed=False).add_to(m)
    return m

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
location_n=sum(valid_point((r['answers'].get('q11') or {}).get('point')) for r in selected)
destination_n=sum(destination_for(r,catalog['places'],False) is not None for r in selected)
print(f'{len(selected)} registros · {location_n} origens com ponto · {destination_n} destinos reconhecidos sem atribuição ilustrativa')
selector=widgets.Dropdown(options=[('Visão geral','')]+[(r['response_id'],r['response_id']) for r in selected],description='Resposta:',layout=widgets.Layout(width='95%'))
demo=widgets.Checkbox(value=False,description='Destinos ilustrativos (somente sintéticos)',indent=False,disabled=NATUREZA!='synthetic',layout=widgets.Layout(width='95%'))
map_output=widgets.Output()
def update_map(change=None):
    with map_output:
        clear_output(wait=True)
        if selector.value:
            r=next(r for r in selected if r['response_id']==selector.value);a=r['answers'];d=destination_for(r,catalog['places'],demo.value)
            display(HTML('<p><b>Destino original:</b> '+html.escape(str(a.get('q15') or 'Não informado'))+'<br><b>Proveniência:</b> '+html.escape(d['provenance'] if d else 'Sem destino reconhecido; nenhum ponto foi inventado')+'</p>'))
        display(territorial_map(selected,catalog,campus,selector.value,demo.value,streets))
selector.observe(update_map,names='value');demo.observe(update_map,names='value')
display(selector,demo,map_output);update_map()

## 8B Diagnósticos e testes de coerência
Conferimos exclusões, classes e intervalos sem apresentar coordenadas. Estes testes detectam erros de processamento; não validam construtos, causalidade ou representatividade. Grupos pequenos reais são suprimidos na tabela de ausências. A análise de sensibilidade por postos é apenas direcional e não é SEM ordinal.

In [ ]:
quality_frame=frames(selected)
columns=['age','confidence','infra','safety','intent','mode']
missing=quality_frame[columns].isna().sum().rename('Ausentes').to_frame()
missing['Percentual']=100*missing['Ausentes']/len(quality_frame)
display(missing)
group_missing=[]
for role,group in quality_frame.groupby('role',dropna=False):
    if NATUREZA!='synthetic' and len(group)<5:continue
    group_missing.append({'Vínculo':role,'n':len(group),'Sem segurança (%)':100*group.safety.isna().mean(),'Sem intenção (%)':100*group.intent.isna().mean()})
display(pd.DataFrame(group_missing))
for model in results['ml']['result']['models']:
    assert sum(map(sum,model['confusion']))==results['ml']['result']['n']
    assert all(0<=point['predicted']<=1 and 0<=point['observed']<=1 for point in model['calibration'])
med=results['mediation']['result'];est=med['estimates']
assert abs(est[2]['estimate']+est[3]['estimate']-est[4]['estimate'])<1e-7
assert all(v['low']<=v['high'] for v in est)
assert all(0<=v['probability']<=1 for v in results['panel']['result']['predictions'])
assert destination_for({'is_synthetic':False,'answers':{'q15':'Destino fictício A'}},catalog['places'],True) is None
assert destination_for({'is_synthetic':True,'answers':{'q15':'local desconhecido'}},catalog['places'],True) is None
fig,ax=plt.subplots(figsize=(7,4))
shown=est[2:]
ax.errorbar([v['estimate'] for v in shown],range(3),xerr=[[v['estimate']-v['low'] for v in shown],[v['high']-v['estimate'] for v in shown]],fmt='o',capsize=4,color='#901812')
ax.set_yticks(range(3),['Indireta','Direta','Total']);ax.axvline(0,color='gray',ls='--');ax.set_title('Associação exploratória • '+NATUREZA);ax.set_xlabel('Escore linear de intenção');plt.tight_layout();plt.show()
print('Coerência computacional verificada. Produto por postos:',med['rank_sensitivity_indirect'])

## 9 Guarde a trilha de reprodução
O arquivo registra filtros, semente, versão do instrumento, quantidade de casos, impressão digital do recorte e versões das bibliotecas. Exporta estimativas, **não respostas individuais**. Ainda assim, revise resultados de grupos pequenos antes de torná-los públicos. Não mantenha saídas de bases reais em cópias públicas do notebook.

In [ ]:
import importlib.metadata as meta
audit={'schema_version':'ufv-mobilidade-3.0.0','natureza':NATUREZA,'n':len(selected),'seed':SEED,'filters':{'role':FILTRO_VINCULO,'mode':FILTRO_MODO,'from':DATA_INICIO,'to':DATA_FIM},'python':platform.python_version(),'libraries':{k:meta.version(k) for k in ['numpy','pandas','scipy','scikit-learn','statsmodels','matplotlib','networkx']},'results':results}
Path('resultados-caminhos-ufv.json').write_text(json.dumps(clean(audit),ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('Arquivo criado: resultados-caminhos-ufv.json. Use o painel de arquivos do Colab para baixar.')

## Referências e próximos passos
- [Validação cruzada no scikit-learn](https://scikit-learn.org/stable/modules/cross_validation.html).
- [GEE no statsmodels](https://www.statsmodels.org/stable/gee.html).
- [Autenticação da planilha com gspread](https://docs.gspread.org/en/latest/oauth2.html).
- [Limitações da mediação transversal — Maxwell e Cole](https://pubmed.ncbi.nlm.nih.gov/17402810/).

Antes de publicação empírica: registrar hipóteses e critérios de exclusão, validar instrumentos, estimar poder por simulação, examinar seleção e faltantes, coletar dados reais e documentar qualquer mudança em relação ao protocolo. Figuras e métricas deste caderno são demonstrações quando a fonte é sintética. Não atribua autoria institucional ou aprovação do instrumento revisado sem verificação.